In [ ]:
import pandas as pd

In [ ]:
cnes = pd.read_csv('arquivos/CNES/cnes_estabelecimentos.csv', sep=';', decimal=',', encoding='latin1', low_memory=False)

In [1]:
import os
import glob
import pandas as pd

In [ ]:
# Pasta contendo os arquivos CSV
pasta_sim = "../arquivos/SIM"

# Lista todos os arquivos CSV, ordenados
arquivos = sorted(glob.glob(os.path.join(pasta_sim, "dados_*.csv")))

# Lista para armazenar os DataFrames filtrados
dfs_filtrados = []

for arquivo in arquivos:
    print(f"Processando: {arquivo}")
    
    # Lê o CSV com separador ponto-e-vírgula e encoding latin1
    df = pd.read_csv(arquivo, sep=";", encoding="latin1", dtype=str)
    
    # Normaliza os nomes das colunas para MAIÚSCULO
    df.columns = df.columns.str.upper()
    
    # Verifica se a coluna TPMORTEOCO existe
    if "TPMORTEOCO" not in df.columns:
        print(f"  ERRO: coluna TPMORTEOCO não encontrada em {arquivo}")
        continue
    
    # Filtra registros com TPMORTEOCO em {1, 2, 3, 4}
    df_filtrado = df[df["TPMORTEOCO"].isin(["1", "2", "3", "4"])]
    
    print(f"  Total registros: {len(df)} | Registros filtrados: {len(df_filtrado)}")
    
    if len(df_filtrado) > 0:
        dfs_filtrados.append(df_filtrado)

# Concatena todos os DataFrames filtrados
if dfs_filtrados:
    df_final = pd.concat(dfs_filtrados, ignore_index=True)
    print(f"\nTotal de registros filtrados (todos os anos): {len(df_final)}")
    
    # Salva o resultado
    # df_final.to_csv("registros_mm.csv", sep=";", index=False, encoding="utf-8-sig")
    print("Arquivo registros_mm.csv salvo com sucesso!")
    print(f"Colunas: {list(df_final.columns)}")
else:
    print("Nenhum registro encontrado para os filtros aplicados.")

In [ ]:
for coluna in df_final.columns:
    if 'ESTAB' in coluna:
        print(f"Coluna encontrada: {coluna}")

In [ ]:
# Verifica se os códigos de estabelecimento de df_final existem no CNES
codigo_estab = df_final['CODESTAB'].astype(str).str.strip()
codigos_cnes = cnes['CO_CNES'].astype(str).str.strip()

mask_existencia = codigo_estab.isin(codigos_cnes)

df_final['codigo_existe_no_cnes'] = mask_existencia

print(f"Total de registros em df_final: {len(df_final)}")
print(f"Registros com código presente no CNES: {mask_existencia.sum()}")
print(f"Registros sem correspondência: {(~mask_existencia).sum()}")

# Opcional: mostrar os códigos sem correspondência
codigos_sem_correspondencia = codigo_estab[~mask_existencia].dropna().unique()
print("Exemplos de códigos sem correspondência:")
print(codigos_sem_correspondencia[:10])

In [ ]:
# Verificação tratanto CODESTAB e CO_CNES como valores numéricos
codigo_estab_num = pd.to_numeric(df_final['CODESTAB'], errors='coerce')
codigos_cnes_num = pd.to_numeric(cnes['CO_CNES'], errors='coerce')

mask_existencia_num = codigo_estab_num.isin(codigos_cnes_num.dropna())

df_final['codigo_existe_no_cnes_num'] = mask_existencia_num

print(f"Total de registros em df_final: {len(df_final)}")
print(f"Registros com código presente no CNES (numérico): {mask_existencia_num.sum()}")
print(f"Registros sem correspondência (numérico): {(~mask_existencia_num).sum()}")

In [ ]:
# Cria chaves numéricas para o join
# Garante que as colunas tenham o mesmo tipo para o merge

df_final_join = df_final.copy()
cnes_join = cnes.copy()

df_final_join['CODESTAB_num'] = pd.to_numeric(df_final_join['CODESTAB'], errors='coerce')
cnes_join['CO_CNES_num'] = pd.to_numeric(cnes_join['CO_CNES'], errors='coerce')

# Remove colunas duplicadas/irrelevantes do CNES para evitar conflitos no merge
cnes_join = cnes_join.drop(columns=['CO_CNES'], errors='ignore')

# Left join
merged_df = (
    df_final_join
    .merge(
        cnes_join,
        left_on='CODESTAB_num',
        right_on='CO_CNES_num',
        how='left',
        suffixes=('_df_final', '_cnes')
    )
)

# Remove a chave auxiliar após o merge
merged_df = merged_df.drop(columns=['CODESTAB_num', 'CO_CNES_num'], errors='ignore')

print(f"Shape do df_final: {df_final.shape}")
print(f"Shape do merged_df: {merged_df.shape}")
print("Colunas adicionadas do CNES:")
print([col for col in merged_df.columns if col.endswith('_cnes') or col in cnes.columns])

In [ ]:
merged_df.to_excel("registros_mm_com_cnes.xlsx", index=False)

In [ ]:
# Conta a quantidade de registros por CODMUNOCOR no dataset unido
contagem_codemunocor = merged_df['CODMUNOCOR'].value_counts()

print(contagem_codemunocor.head())
print(f"\nCODMUNOCOR com maior quantidade de registros: {contagem_codemunocor.idxmax()} ({contagem_codemunocor.max()} registros)")

In [2]:
import os
import glob
import pandas as pd
import re

# Dicionários para armazenar cabeçalhos: {fonte: {ano: [colunas]}}
headers = {"SIM": {}, "SINASC": {}}

# --- SIM ---
pasta_sim = "../arquivos/SIM"
for arquivo in sorted(glob.glob(os.path.join(pasta_sim, "dados_*.csv"))):
    ano_match = re.search(r"dados_(\d{4})\.csv", arquivo)
    ano = int(ano_match.group(1))
    with open(arquivo, "r", encoding="latin1") as f:
        linha = f.readline().strip()
    colunas = [c.strip().strip('"').upper() for c in linha.split(";")]
    headers["SIM"][ano] = colunas

# --- SINASC ---
pasta_sinasc = "../arquivos/SINASC"
for arquivo in sorted(glob.glob(os.path.join(pasta_sinasc, "SINASC_*.csv"))):
    ano_match = re.search(r"SINASC_(\d{4})\.csv", arquivo)
    ano = int(ano_match.group(1))
    with open(arquivo, "r", encoding="latin1") as f:
        linha = f.readline().strip()
    colunas = [c.strip().strip('"').upper() for c in linha.split(";")]
    headers["SINASC"][ano] = colunas

# Coleta todas as colunas únicas
todas_colunas = set()
for fonte in headers:
    for ano in headers[fonte]:
        todas_colunas.update(headers[fonte][ano])

todas_colunas = sorted(todas_colunas)

# Constrói o DataFrame com índice múltiplo (ano, fonte)
registros = []
for fonte in ["SIM", "SINASC"]:
    for ano in sorted(headers[fonte].keys()):
        cols_arquivo = set(headers[fonte][ano])
        row = {col: (col in cols_arquivo) for col in todas_colunas}
        row["ano"] = ano
        row["fonte"] = fonte
        registros.append(row)

df_colunas = pd.DataFrame(registros)
df_colunas = df_colunas.set_index(["ano", "fonte"])
df_colunas.index.names = ["Ano", "Fonte"]

print(f"Shape do dataset: {df_colunas.shape}")
print(f"Total de colunas únicas: {len(todas_colunas)}")
print(f"Total de arquivos: {len(registros)}")
print("\nAmostra (primeiras 5 colunas, primeiros 6 registros):")
display(df_colunas.iloc[:6, :5])

# Mostra colunas que NÃO estão presentes em todos os arquivos
presenca = df_colunas.sum(axis=0)
colunas_variaveis = presenca[presenca < len(registros)]
print(f"\nColunas com variação de presença ({len(colunas_variaveis)} de {len(todas_colunas)}):")
for col, count in colunas_variaveis.sort_values().items():
    ausentes = df_colunas[~df_colunas[col]].index.tolist()
    print(f"  {col}: presente em {int(count)}/{len(registros)} - ausente em: {ausentes}")


Shape do dataset: (22, 132)
Total de colunas únicas: 132
Total de arquivos: 22

Amostra (primeiras 5 colunas, primeiros 6 registros):


,,ACIDTRAB,ALTCAUSA,APGAR1,APGAR5,ASSISTMED
Ano,Fonte,,,,,
2013,SIM,True,False,False,False,True
2014,SIM,True,True,False,False,True
2015,SIM,True,True,False,False,True
2016,SIM,True,True,False,False,True
2017,SIM,True,True,False,False,True
2018,SIM,True,True,False,False,True



Colunas com variação de presença (111 de 132):
  CODCART: presente em 1/22 - ausente em: [(2013, 'SIM'), (2014, 'SIM'), (2015, 'SIM'), (2016, 'SIM'), (2017, 'SIM'), (2018, 'SIM'), (2019, 'SIM'), (2020, 'SIM'), (2021, 'SIM'), (2022, 'SIM'), (2023, 'SIM'), (2014, 'SINASC'), (2015, 'SINASC'), (2016, 'SINASC'), (2017, 'SINASC'), (2018, 'SINASC'), (2019, 'SINASC'), (2020, 'SINASC'), (2021, 'SINASC'), (2022, 'SINASC'), (2023, 'SINASC')]
  CB_ALT: presente em 1/22 - ausente em: [(2013, 'SIM'), (2014, 'SIM'), (2015, 'SIM'), (2016, 'SIM'), (2017, 'SIM'), (2018, 'SIM'), (2019, 'SIM'), (2020, 'SIM'), (2021, 'SIM'), (2022, 'SIM'), (2013, 'SINASC'), (2014, 'SINASC'), (2015, 'SINASC'), (2016, 'SINASC'), (2017, 'SINASC'), (2018, 'SINASC'), (2019, 'SINASC'), (2020, 'SINASC'), (2021, 'SINASC'), (2022, 'SINASC'), (2023, 'SINASC')]
  RACACORN: presente em 1/22 - ausente em: [(2013, 'SIM'), (2014, 'SIM'), (2015, 'SIM'), (2016, 'SIM'), (2017, 'SIM'), (2018, 'SIM'), (2019, 'SIM'), (2020, 'SIM'), (2021, 'SI

In [ ]:
# Verifica o comprimento dos valores de CODMUNOCOR no dataset unido
codmun = merged_df['CODMUNOCOR'].astype(str).str.strip()
comprimentos = codmun.str.len()

print('Resumo do comprimento de CODMUNOCOR:')
print(comprimentos.value_counts().sort_index())
print(f"\nQuantidade com 6 dígitos: {(comprimentos == 6).sum()}")
print(f"Quantidade com 7 dígitos: {(comprimentos == 7).sum()}")
print(f"Quantidade com mais de 7 dígitos: {(comprimentos > 7).sum()}")

if (comprimentos == 7).sum() > 0:
    print("\nExemplos com 7 dígitos:")
    print(codmun[comprimentos == 7].dropna().head().tolist())

In [ ]:
# Filtra merged_df para o município de São Paulo (CODMUNOCOR = 355030)
merged_df_sp = merged_df[merged_df['CODMUNOCOR'] == '355030'].copy()

print(f"Quantidade de registros após o filtro: {len(merged_df_sp)}")
print(merged_df_sp.head())

In [ ]:
merged_df_sp.to_csv("registros_mm_sp.csv", sep=";", index=False, encoding="utf-8-sig")

In [3]:
df_colunas.to_csv("colunas_variaveis.csv", sep=";", index=True, encoding="utf-8-sig")